In [1]:
import pandas as pd
import sienna
from evaluation import SchemaIntegrationEvaluation
import os
import numpy as np

### Analysis

In [ ]:
# How many times were the loops used?
loops_per_use_case = {}
for file_name in os.listdir("logs/"):
    # if ".json" in file_name and "Qwen" in file_name and "Real Benchmark" not in file_name:
    if ".json" in file_name and "gpt-5.2" in file_name and "Real Benchmark" not in file_name:
        log_file_name = "logs/" + file_name.replace("_evaluation.json", "").replace(".json", "")
        schema_evaluator = SchemaIntegrationEvaluation(tables_path="data/selected-tables/SINT-Benchmark", log_file_name=log_file_name)
        if schema_evaluator.self_consistency:
            if schema_evaluator.run_info["sequence_of_phases"] == [ "detect_tables_phase", "schema_matching_phase", "grouping_phase", "schema_integration_phase", "final_integration_phase"] and schema_evaluator.self_consistency:
            # if schema_evaluator.run_info["sequence_of_phases"] == ["schema_matching_phase", "schema_integration_phase", "detect_tables_phase", "final_integration_phase"] :
                schema_evaluator.run_evaluation()
                use_case = schema_evaluator.run_info["folder_name"]
                if use_case not in loops_per_use_case:
                    loops_per_use_case[use_case] = {"detect_tables_phase": 0, "grouping_phase": 0, "final_integration_phase": 0}
                for run in schema_evaluator.all_predictions:
                    if "detect_tables_phase" in schema_evaluator.all_predictions[run]:
                        loops_per_use_case[use_case]["detect_tables_phase"] += sum(schema_evaluator.all_predictions[run]["detect_tables_phase"]["missing_loop"].values())
                    if "grouping_phase" in schema_evaluator.all_predictions[run]:
                        loops_per_use_case[use_case]["grouping_phase"] += schema_evaluator.all_predictions[run]["grouping_phase"]["missed_tables_reassigned"]
                    if "final_integration_phase" in schema_evaluator.all_predictions[run] and "missed_attributes_loop" in schema_evaluator.all_predictions[run]["final_integration_phase"]:
                        loops_per_use_case[use_case]["final_integration_phase"] += schema_evaluator.all_predictions[run]["final_integration_phase"]["missed_attributes_loop"]

In [ ]:
# How is the schema matching loop change the integrated schema/mappings results?
schema_integration_res = {}
all_schema_evaluators = {}
for file_name in os.listdir("logs/"):
    # if ".json" in file_name and "Qwen" in file_name and "Real Benchmark" not in file_name:
    if ".json" in file_name and "gpt-5.2" in file_name and "Real Benchmark" not in file_name:
        log_file_name = "logs/" + file_name.replace("_evaluation.json", "").replace(".json", "")
        schema_evaluator = SchemaIntegrationEvaluation(tables_path="data/selected-tables/SINT-Benchmark", log_file_name=log_file_name)
        if schema_evaluator.self_consistency:
            if schema_evaluator.run_info["sequence_of_phases"] == [ "detect_tables_phase", "schema_matching_phase", "grouping_phase", "schema_integration_phase", "final_integration_phase"] and schema_evaluator.self_consistency:
            # if schema_evaluator.run_info["sequence_of_phases"] == ["schema_matching_phase", "schema_integration_phase", "detect_tables_phase", "final_integration_phase"] :
                schema_evaluator.run_evaluation()
                eval_results = schema_evaluator.eval_results
                use_case = schema_evaluator.run_info["folder_name"]
                all_schema_evaluators[use_case] = schema_evaluator
                schema_integration_res[use_case] = eval_results["0"]["schema_integration_phase_extended"]


In [ ]:
# Integrated schemas analysis
rows = []
for use_case in schema_integration_res:
    removed = schema_integration_res[use_case]["integrated_schemas"]["schema_integration_phase_schema"]["eval_results_attributes"]["eval_stats"]
    if "integrated_attributes_no_removals" in schema_integration_res[use_case]:
        not_removed = schema_integration_res[use_case]["integrated_attributes_no_removals"]["schema_integration_phase_schema_no_removals"]["eval_results_attributes"]["eval_stats"]
    else:
        not_removed = removed
        # print(f"No separate evaluation for no removals in use case {use_case}, using the same results as with removals.")
    rows.append([use_case, removed["overall_precision"], removed["overall_recall"], removed["overall_f1"], not_removed["overall_precision"], not_removed["overall_recall"], not_removed["overall_f1"]])

In [ ]:
# Mappings analysis
rows = []
for use_case in schema_integration_res:
    removed = schema_integration_res[use_case]["integrated_schemas"]["schema_integration_phase_column_mappings"]
    if "integrated_attributes_no_removals" in schema_integration_res[use_case]:
        not_removed = schema_integration_res[use_case]["integrated_attributes_no_removals"]["schema_integration_phase_column_mappings"]
    else:
        not_removed = removed
        # print(f"No separate evaluation for no removals in use case {use_case}, using the same results as with removals.")
    rows.append([use_case, removed["overall_precision"], removed["overall_recall"], removed["overall_f1"], not_removed["overall_precision"], not_removed["overall_recall"], not_removed["overall_f1"]])

In [4]:
results_df = pd.DataFrame(rows, columns=["use_case", "precision_with_removals", "recall_with_removals", "f1_with_removals", "precision_no_removals", "recall_no_removals", "f1_no_removals"])

In [6]:
print("Average metrics with removals:")
print(f"Precision: {results_df['precision_with_removals'].mean()}")
print(f"Recall: {results_df['recall_with_removals'].mean()}")
print(f"F1: {results_df['f1_with_removals'].mean():.4f}")

Average metrics with removals:
Precision: 0.9334028832109027
Recall: 0.9334028832109027
F1: 0.9334


In [7]:
print("Average metrics without removals:")
print(f"Precision: {results_df['precision_no_removals'].mean()}")
print(f"Recall: {results_df['recall_no_removals'].mean()}")
print(f"F1: {results_df['f1_no_removals'].mean()}")

Average metrics without removals:
Precision: 0.8202896173481842
Recall: 0.8202896173481842
F1: 0.8202896173481842
